# Veritas-R1 — Stage 3: Iterative SimPO (Meta-aligned preference tuning)

Replaces the planned DPO stage with Meta's Llama 3.1 recipe: **3 rounds of SimPO** with fresh preference pairs sampled from the previous round's checkpoint.

**Why SimPO over DPO** (Meng et al. 2024 — adopted in Llama 3.1 post-training):
- No reference model → ~50% GPU memory saved → larger batch on Kaggle T4
- Length-normalised reward → no length bias
- Beats DPO by 3–7 points on AlpacaEval2 / Arena-Hard / MT-Bench

**Why iterative**: each round trains on *fresh* (chosen, rejected) pairs sampled from the current best checkpoint, then scored. Llama 3.1 paper §4 reports 3 rounds beating single-pass DPO by 4–7 points on Arena-Hard.

**Inputs**: a Stage-1 SFT adapter (from `veritas-stage1-train.ipynb`) at `/kaggle/working/adapters/seed_17/adapter`.
**Outputs**: three round adapters at `/kaggle/working/simpo/round-{1,2,3}/adapter`.

In [ ]:
%pip install --quiet --upgrade \
    "transformers>=4.46,<5" \
    "peft>=0.13" \
    "trl>=0.11" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    "bitsandbytes>=0.44"

In [ ]:
import os, json, math, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset, Dataset

BASE_MODEL = "Qwen/Qwen3-1.7B-Instruct"
OUT_ROOT = Path("/kaggle/working")
STAGE1_ADAPTER = OUT_ROOT / "adapters" / "seed_17" / "adapter"  # change if a different seed is preferred
SIMPO_ROOT = OUT_ROOT / "simpo"
SIMPO_ROOT.mkdir(parents=True, exist_ok=True)

ROUNDS = 3
PAIRS_PER_ROUND = 5_000
GENERATIONS_PER_PROMPT = 4   # sample 4, keep best+worst as (chosen, rejected)
MAX_NEW_TOKENS = 256
TEMP = 0.85
TOP_P = 0.92

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device =", device, "| torch =", torch.__version__)

## 1. Prompt pool for sampling fresh pairs

We pull a workspace-shaped pool of prompts from SmolTalk-2 + Tulu-3 + OpenThoughts. Each round samples a fresh slice so successive rounds see different prompts.

In [ ]:
def collect_prompts(target=20_000):
    pool = []
    sources = [
        ("HuggingFaceTB/smol-talk-2", None, "messages"),
        ("allenai/tulu-3-sft-mixture", None, "messages"),
    ]
    for hf, cfg, key in sources:
        kwargs = {"streaming": True}
        if cfg: kwargs["name"] = cfg
        ds = load_dataset(hf, split="train", **kwargs)
        for row in ds:
            msgs = row.get(key)
            if not isinstance(msgs, list) or not msgs: continue
            user_msgs = [m for m in msgs if isinstance(m, dict) and m.get("role") == "user" and m.get("content")]
            if not user_msgs: continue
            pool.append({"prompt": str(user_msgs[0]["content"])[:2000], "source": hf})
            if len(pool) >= target: break
        if len(pool) >= target: break
    rng = random.Random(17)
    rng.shuffle(pool)
    print(f"prompt pool: {len(pool)}")
    return pool

PROMPT_POOL = collect_prompts(target=ROUNDS * PAIRS_PER_ROUND * 2)

## 2. Generation helper — load adapter, sample N completions per prompt

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

def load_for_generation(adapter_path):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2" if device=="cuda" else "eager")
    model = PeftModel.from_pretrained(base, str(adapter_path))
    model.eval()
    return tok, model

@torch.no_grad()
def sample_completions(tok, model, prompt, n=GENERATIONS_PER_PROMPT, max_new=MAX_NEW_TOKENS):
    msgs = [
        {"role": "system", "content": "You are Veritas-R1, the AI inside Forge. Be direct, accurate, and verifiable."},
        {"role": "user", "content": prompt},
    ]
    input_ids = tok.apply_chat_template(msgs, return_tensors="pt", add_generation_prompt=True).to(device)
    completions = []
    for _ in range(n):
        out = model.generate(input_ids, max_new_tokens=max_new, do_sample=True, temperature=TEMP, top_p=TOP_P, pad_token_id=tok.pad_token_id)
        completion = tok.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True).strip()
        completions.append(completion)
    return completions

## 3. Pair scorer — rule-based judge

A simple but effective heuristic: completions get scored by **length sanity** + **structural format** + **lexical novelty**. Highest-scoring becomes `chosen`, lowest becomes `rejected`. For production this should be replaced with a reward model (Llama 3.1 uses one); the rule-based judge gets us iterating without that dependency.

In [ ]:
def score_completion(c):
    if not c or not c.strip(): return -1.0
    L = len(c)
    # Penalise too short or too long
    length_score = 1.0 if 80 <= L <= 1800 else (0.5 if 40 <= L <= 2400 else 0.1)
    # Bonus if it has structure (headings/lists/<think>)
    structure_score = 0.0
    if "<think>" in c and "</think>" in c: structure_score += 0.3
    if c.count("\n\n") >= 1: structure_score += 0.15
    if c.startswith(("#", "-", "1.")): structure_score += 0.1
    # Penalise repetition
    words = c.split()
    novelty = (len(set(words)) / max(len(words), 1))
    return length_score + structure_score + 0.6 * novelty

def make_pair(prompt, completions):
    scored = sorted(((score_completion(c), c) for c in completions), key=lambda x: x[0], reverse=True)
    if scored[0][0] - scored[-1][0] < 0.15: return None  # too close → discard
    return {"prompt": prompt, "chosen": scored[0][1], "rejected": scored[-1][1]}

## 4. Build a round of preference data

In [ ]:
def build_preference_data(adapter_path, prompts, target=PAIRS_PER_ROUND, out_path=None):
    print(f"building preferences from {adapter_path} | target={target} pairs")
    tok, model = load_for_generation(adapter_path)
    pairs = []
    seen = 0
    for p in prompts:
        seen += 1
        comps = sample_completions(tok, model, p["prompt"])
        pair = make_pair(p["prompt"], comps)
        if pair is not None:
            pairs.append(pair)
            if len(pairs) % 250 == 0:
                print(f"  pairs={len(pairs)} / target={target} (saw={seen})")
        if len(pairs) >= target: break
    del model; gc.collect(); torch.cuda.empty_cache()
    df = pd.DataFrame(pairs)
    if out_path is not None:
        df.to_parquet(out_path, index=False)
    print(f"  built {len(df)} pairs → {out_path}")
    return df

## 5. SimPO trainer — TRL

Uses TRL's `SimPOTrainer` (≥ 0.11). Key hyperparams from Meng et al. 2024:
- β = 2.5 (reward scale)
- γ = 1.4 (target margin)
- LR = 5e-7 (much lower than SFT — we're only nudging the policy)

In [ ]:
from trl import SimPOConfig, SimPOTrainer
from peft import LoraConfig

def train_simpo_round(round_idx, sft_adapter_path, pair_parquet, out_dir):
    print(f"\n=== SimPO round {round_idx} ===")
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb, torch_dtype=torch.bfloat16, attn_implementation="flash_attention_2" if device=="cuda" else "eager")
    model = PeftModel.from_pretrained(base, str(sft_adapter_path), is_trainable=True)
    
    ds = load_dataset("parquet", data_files={"train": str(pair_parquet)}, split="train")
    
    cfg = SimPOConfig(
        output_dir=str(out_dir),
        num_train_epochs=1.0,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=5e-7,
        warmup_ratio=0.1,
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        bf16=True,
        gradient_checkpointing=True,
        beta=2.5,
        gamma_beta_ratio=1.4 / 2.5,
        max_length=1024,
        max_prompt_length=512,
        logging_steps=25,
        save_strategy="epoch",
        seed=17 + round_idx,
        optim="adamw_torch",
        report_to=[],
        remove_unused_columns=False,
    )
    new_lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=["q_proj","k_proj","v_proj","o_proj"])
    trainer = SimPOTrainer(model=model, args=cfg, train_dataset=ds, processing_class=tok, peft_config=new_lora)
    out = trainer.train()
    adapter_dir = out_dir / "adapter"
    trainer.model.save_pretrained(adapter_dir)
    tok.save_pretrained(adapter_dir)
    summary = {"round": round_idx, "loss": float(out.training_loss) if out.training_loss else None, "steps": int(out.global_step), "pairs": len(ds)}
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2))
    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    print(f"  round {round_idx} done: {summary}")
    return summary

## 6. Run all 3 rounds

Each round consumes a fresh slice of the prompt pool, samples completions from the latest checkpoint, builds preference pairs, and trains. Round N+1 starts from round N's adapter.

In [ ]:
current_adapter = STAGE1_ADAPTER
round_summaries = []
for r in range(1, ROUNDS + 1):
    round_dir = SIMPO_ROOT / f"round-{r}"
    round_dir.mkdir(parents=True, exist_ok=True)
    pair_path = round_dir / "pairs.parquet"
    
    # 1. Build pairs from current adapter (skip if already built)
    if not pair_path.is_file():
        slice_start = (r - 1) * PAIRS_PER_ROUND * 2
        slice_prompts = PROMPT_POOL[slice_start: slice_start + PAIRS_PER_ROUND * 2]
        build_preference_data(current_adapter, slice_prompts, target=PAIRS_PER_ROUND, out_path=pair_path)
    else:
        print(f"round {r}: pairs cached at {pair_path}")
    
    # 2. Train SimPO on these pairs
    if not (round_dir / "adapter").is_dir():
        s = train_simpo_round(r, current_adapter, pair_path, round_dir)
        round_summaries.append(s)
    else:
        print(f"round {r}: adapter cached")
        round_summaries.append(json.loads((round_dir / "summary.json").read_text()))
    
    current_adapter = round_dir / "adapter"

(SIMPO_ROOT / "simpo_run.json").write_text(json.dumps({"rounds": round_summaries, "final_adapter": str(current_adapter)}, indent=2))
print("\nAll rounds done.")
for r in round_summaries:
    print(f"  round {r['round']}: loss={r['loss']:.4f} pairs={r['pairs']}")

## 7. Next step — run `veritas-stage1-eval.ipynb` against the final adapter

Point the eval notebook's `ADAPTER_ROOT` at `/kaggle/working/simpo/round-3/adapter` to score the SimPO-tuned model on ForgeBench-Reason and produce reliability + ECE plots.